# Project 85 — Local Image Captioning Pipeline
## Scene Description → Detailed Caption → Alt-Text → SEO Tags

**Stack:** LangChain · Ollama · Pydantic · Jupyter

*Note: Uses text-based scene simulation since Ollama doesn't support vision natively.*

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Scene Descriptions (simulating image input)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

scenes = [
    {"id": "img_001", "description": "A golden retriever playing fetch on a sandy beach at sunset, "
     "waves crashing in the background, owner visible in the distance"},
    {"id": "img_002", "description": "A busy coffee shop interior with exposed brick walls, "
     "people working on laptops, barista making latte art, warm lighting"},
    {"id": "img_003", "description": "A mountain landscape with snow-capped peaks, "
     "pine forest in the foreground, a clear blue lake reflecting the mountains"},
    {"id": "img_004", "description": "A modern kitchen with marble countertops, someone "
     "preparing a colorful salad, fresh vegetables on a cutting board"},
    {"id": "img_005", "description": "A street market in Asia with colorful food stalls, "
     "lanterns hanging overhead, crowds of people browsing"},
]
print(f"Scenes to caption: {len(scenes)}")

## Step 2 — Multi-Format Caption Generation

In [ ]:
class ImageCaptions(BaseModel):
    short_caption: str = Field(description="1 sentence, 10-15 words")
    detailed_caption: str = Field(description="2-3 sentences with scene details")
    alt_text: str = Field(description="Accessible alt text for screen readers, 125 chars max")
    seo_tags: list[str] = Field(description="5-8 SEO keywords/tags")
    mood: str = Field(description="emotional tone: warm, serene, energetic, etc.")
    color_palette: list[str] = Field(description="3-5 dominant colors")

captioner = llm.with_structured_output(ImageCaptions)

caption_results = []
for scene in scenes:
    captions = captioner.invoke(
        f"Generate captions for an image showing: {scene['description']}"
    )
    caption_results.append({"id": scene["id"], "scene": scene["description"], "captions": captions})
    print(f"\n{scene['id']}:")
    print(f"  Short: {captions.short_caption}")
    print(f"  Alt: {captions.alt_text[:80]}...")
    print(f"  Tags: {captions.seo_tags[:5]}")
    print(f"  Mood: {captions.mood} | Colors: {captions.color_palette}")

## Step 3 — Social Media Post Generator

In [ ]:
social_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a social media post for this image. Include relevant hashtags. "
     "Match the platform style."),
    ("human", "Image: {description}\nPlatform: {platform}\nPost:")
])
social_chain = social_prompt | llm | StrOutputParser()

platforms = ["Instagram", "Twitter", "LinkedIn"]
for scene in scenes[:3]:
    print(f"\n{'='*50}")
    print(f"Scene: {scene['description'][:50]}...")
    for platform in platforms:
        post = social_chain.invoke({
            "description": scene["description"],
            "platform": platform,
        })
        print(f"\n  [{platform}] {post[:120]}...")

## Step 4 — Accessibility Report

In [ ]:
print("ACCESSIBILITY REPORT")
print("=" * 50)
for cr in caption_results:
    alt = cr["captions"].alt_text
    issues = []
    if len(alt) > 125:
        issues.append(f"Alt text too long ({len(alt)} chars, max 125)")
    if len(alt) < 20:
        issues.append("Alt text too short")
    if not any(c.isalpha() for c in alt):
        issues.append("Alt text has no descriptive words")

    status = "✓" if not issues else "⚠"
    print(f"  {status} {cr['id']}: {alt[:60]}... ({len(alt)} chars)")
    for issue in issues:
        print(f"    → {issue}")

# Export
from pathlib import Path
Path("sample_data").mkdir(exist_ok=True)
export = [{"id": cr["id"], **cr["captions"].model_dump()} for cr in caption_results]
with open("sample_data/image_captions.json", "w") as f:
    json.dump(export, f, indent=2)
print(f"\n✓ Exported {len(export)} captions")

## What You Learned
- **Multi-format captioning** (short, detailed, alt-text, SEO)
- **Social media adaptation** per platform
- **Accessibility guidelines** enforcement
- **Structured metadata** from visual content